In [ ]:
!pip install mlflow

In [ ]:
import mlflow
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import make_judge
from openai import OpenAI
from typing import Literal
import os, re, json

In [ ]:
LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:80"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required", #for now
)

In [ ]:
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        os.environ["MLFLOW_WORKSPACE"] = f.read().strip()

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("canopy-backend")

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace: {os.environ.get('MLFLOW_WORKSPACE', 'not set')}")

## Add to the dataset with external expectations and inputs

In [ ]:
eval_dataset = mlflow.genai.get_dataset(
    name="eval_dataset",
)

In [ ]:
new_samples = [                                                                                                                                                                                                                        
    {                                                                                                                                                                                                                                  
        "inputs": {                                                                                                                                                                                                                  
            "messages": [
                {"role": "user", "content": "Artificial intelligence is transforming healthcare by enabling faster diagnostics and personalized treatment plans."},
            ],
            "session_id": "session-001",
        },
        "expectations": {"length": 150},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "The Eiffel Tower was built in 1889 and stands 330 metres tall in Paris, France."},
                {"role": "assistant", "content": "The Eiffel Tower, built in 1889, is a 330-metre landmark in Paris. 🗼"},
                {"role": "user", "content": "What year was it built again?"},
            ],
            "session_id": "session-002",
        },
        "expectations": {"length": 100},
    },
]

eval_dataset.merge_records(new_samples)
print(f"Dataset now has records: {eval_dataset.to_dict()['profile']}")

## Define some scorers to use for evaluation

In [ ]:
@scorer
def is_concise(outputs: str, expectations: dict) -> bool:
    """Is the response under the character limit?"""
    if not outputs:
        return False
    return len(outputs) <= expectations.get("length", 300)

os.environ["OPENAI_API_KEY"] = "no-key-required"

summary_quality_judge = make_judge(
    name="summary_quality",
    instructions=(
        "Evaluate the quality of the GENERATED_RESPONSE given the CONVERSATION history.\n\n"
        "{{ inputs }}\n"
        "{{ outputs }}\n\n"
        "Is the response a concise, accurate, and helpful reply to the conversation?"
    ),
    feedback_value_type=Literal["yes", "no"],
    model="openai:/llama32",
    base_url="http://llama-32-predictor.ai501.svc.cluster.local:80/v1/chat/completions",
    extra_headers={"Authorization": "Bearer no-key-required"},
)


print("✓ Scorers defined: summary_quality, is_concise")

## Run evaluation

We run the evaluation against the backend endpoint to make sure that we evaluate against the exact prompt and code we have in the backend

In [ ]:
def send_request(payload, url):
    import httpx
    import json
    full_response = ""

    with httpx.Client(timeout=None) as client:
        with client.stream("POST", url, json=payload) as response:
            for line in response.iter_lines():
                if line.startswith("data: "):
                    try:
                        data = json.loads(line[len("data: "):])
                        full_response += data.get("delta", "")
                    except json.JSONDecodeError:
                        continue

    return full_response

def predict_backend_summarize(messages: list, session_id: str = None, **kwargs) -> str:
      from urllib.parse import urljoin
      backend_url = "http://canopy-backend:8000"
      endpoint_to_test = "/summarize/chat"

      url = urljoin(backend_url, endpoint_to_test)
      payload = {
          "messages": messages,
          "session_id": session_id,
      }
      return send_request(payload, url)

In [ ]:
print("Running evaluation with our latest prompt...")
eval_v1 = mlflow.genai.evaluate(
    data=eval_dataset,      # ← using the MLflow-hosted dataset
    predict_fn=predict_backend_summarize,
    scorers=[is_concise, summary_quality_judge],
)

print("\n── Evaluation Results: Prompt v1 ──")
for metric, value in sorted(eval_v1.metrics.items()):
    bar = "█" * int((value if isinstance(value, float) else 0) * 20)
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}  {bar}")

print(f"\n✓ Results saved. View per-example breakdowns in the Evaluations tab.")
print(f"  → {MLFLOW_TRACKING_URI}")